___
# Dataset EDA and preparation

In [1]:
import numpy as np
import pandas as pd
from utils import COLUMN_DEFINITIONS

## Load synthetic data

In [2]:
# Load t2dm dataset
df = pd.read_csv('../data/t2dm_synthetic_T2DM_only.csv')
df

,hospital,pkey,age,sex,interpreter,hypertension,hyperlipidaemia,ischemic_heart_disease,cardiac_failure,neuropathy,...,neuropathy_ep,visits_b2025,method_manage_dt2,ind_of_nephropathy,cardiovascular_disease,cerebrovascular_disease,lower_limb_problems,admissions_2024,outcome,height_m
0,WH,922904,18.000000,F,0,0,0,0,0,0,...,0,2.000000,NON-INSULIN,0,0,0,0,0.616392,0,1.726853
1,RMH,118725,19.000000,M,0,0,0,0,0,1,...,0,3.000000,TABLETS,0,0,0,0,0.000000,0,1.582828
2,RMH,888078,19.000000,F,0,0,0,0,0,0,...,0,14.000000,DIET ONLY,0,0,0,0,12.164862,0,1.692531
3,WH,811787,20.000000,M,0,0,0,1,0,0,...,0,2.000000,INSULIN,0,1,0,0,0.899655,0,1.672373
4,WH,334671,28.324544,F,1,0,0,0,0,0,...,0,12.406219,TABLETS,0,0,0,0,0.000000,1,1.620002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3822,WH,695216,86.000000,M,0,1,1,0,1,0,...,0,2.000000,UNKNOWN,0,0,0,0,2.538218,0,1.796923
3823,WH,694586,87.578208,F,0,0,0,1,0,0,...,0,13.406219,BOTH INSULIN AND TABS,0,0,0,1,8.198681,1,1.694737
3824,WH,161717,86.010532,M,0,0,0,0,0,1,...,0,14.406219,BOTH INSULIN AND TABS,0,0,0,0,1.002093,1,1.670805
3825,WH,855731,86.000000,F,0,0,0,0,0,0,...,0,49.000000,TABLETS,0,0,0,0,0.000000,0,1.623093


## Rename and reorder

In [3]:
df.rename(columns={
    'pkey': 'patient_id',
    'calcium_chan_block': 'calcium_channel_blocker',
    'agr2receptorblock': 'agr2_receptor_blocker',
    'familyhistory': 'family_history',
    'gestdiabetes': 'gestational_diabetes',
    'symptomaticpn': 'sympt_peripheral_neuropathy',
    'isletantibody': 'islet_antibody',
    'coeliacantibody': 'coeliac_antibody',
    'tfunction': 'thyroid_function',
    'tantibody': 'thyroid_antibody',
    'vitaminb12': 'vitamin_b12',
    'wcc': 'white_cell_count',
    'rcc': 'red_cell_count',
    'pcv': 'packed_cell_volume',
    'mcv': 'mean_cell_volume',
    'weight_more_100': 'weight_over_100',
    'systolic': 'systolic_bp',
    'diastolic': 'diastolic_bp',
    'charcot': 'charcot_foot',
    'ami': 'acute_myocardial_infarction',
    'tia': 'transient_ischemic_attack',
    'neuropathy_ep': 'autonomic_neuropathy',
    'visits_b2025': 'visits_before_2025',
    'method_manage_dt2': 'method_manage_t2dm',
    'ind_of_nephropathy': 'nephropathy_indication',
    'admissions_2024': 'admissions_in_2024',
    'height_m': 'height',
    'outcome': 'outcome_admission_in_2025',
    'years_diagnosed': 'years_since_diagnosis',
    }, inplace=True)

In [4]:
df = df[COLUMN_DEFINITIONS.keys()].copy()

## Correct column values

### Convert height to cm and add "data-entry" errors

In [5]:
# Convert height to cm and round to 2 decimal places
df.height = df.height.round(2) * 100

# # Introduce some errors in the height column
# idx = df.sample(5, random_state=43).index
# df.loc[idx, 'height'] = [16, 1810, 76, 119, 1780]

### Introduce errors in recorded weight and update flags for weight > 100kg

In [6]:
# Introduce some errors in the weight column
idx = df.sample(4, random_state=44).index
df.loc[idx, 'weight'] = [789, 10.49, 668, 709]

# Set weight_over_100 to 1 for patients with weight > 100kg, and 0 otherwise
df['weight_over_100'] = (df.weight > 100).astype(int)

### Re-calculate BMI, update the flag for obesity, then randomly remove some BMI values

In [7]:
# Recalculate BMI based on height and weight values
df['bmi'] = df.apply(lambda x: x.weight / (x.height / 100) ** 2, axis=1)

# Set obesity to 1 for patients with BMI >= 30, and 0 otherwise
df['obesity'] = (df.bmi >= 30).astype(int)

# Randomaly remove some BMI values
idx = df.sample(204, random_state=42).index
df.loc[idx, 'bmi'] = np.nan

### Convert age to integer, introduce "data-entry" errors, and group over 86yo

In [8]:
# Convert to integer
df.age = df.age.astype(int)

# Introduce some errors in the age column
idx = df.sample(4, random_state=45).index
df.loc[idx, 'age'] = [5, 609, 4, 17]

# Group ages above 86 into a single category
df.loc[df.age >= 86, 'age'] = 86

### Convert to integer the number of past visits/admissions

In [9]:
# Convert to integer
df.admissions_in_2024 = df.admissions_in_2024.astype(int)
df.visits_before_2025 = df.visits_before_2025.astype(int)

### Make sure the number of years since the person has been diagnosed is less than their age

In [10]:
# All visits before 2025 were in 2024, so years_since_diagnosis should be equal to 1
# df.loc[df.visits_before_2025==df.admissions_in_2024, 'years_since_diagnosis'] = 1

In [11]:
# Generate years_since_diagnosis as a random integer between 1 and half of the patient's age
df['years_since_diagnosis'] = df.age.apply(lambda x: np.random.randint(1, x/2)).astype(int)

### Flag for oral contraceptives should be False in men

In [12]:
# Set to 0 oral_contraceptive use for male patients, as this is not applicable
df.loc[(df.oral_contraceptive == 1) & (df.sex == 'M'), 'oral_contraceptive'] = 0

### Introduce errors in white cell counts

In [13]:
idx = df.sample(1, random_state=46).index
df.loc[idx, 'white_cell_count'] = 1006

### Round to 1 decimal point

In [14]:
cols = ['weight', 'waist', 'systolic_bp', 'diastolic_bp',
        'sodium', 'potassium', 'chloride', 'bicarbonate', 
        'creatinine', 'urea', 'haemoglobin', 'albumin', 
        'white_cell_count', 'platelets', 'red_cell_count', 
        'packed_cell_volume', 'mean_cell_volume', 'hba1c', 'egfr']

df[cols] = df[cols].round(1)

### Convert column types

In [15]:
# Boolean columns
bool_cols = [col for col in df.columns if (df[col].dtype == 'int64') and (df[col].nunique() <= 2)]
df[bool_cols] = df[bool_cols].astype('bool')

# Categorical columns
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
df[cat_cols] = df[cat_cols].astype('category')

df.dtypes.value_counts()

bool        46
float64     21
int64        5
category     1
category     1
category     1
category     1
Name: count, dtype: int64

## Save the data

In [16]:
# Save to a .parquet file to preserve data types
df.to_parquet("../data/t2dm_synthetic_T2DM_only_prepared.parquet", engine="pyarrow")